# Exploratory Data Analysis - Weather Forecast Pipeline

Data: the **silver layer** `clean.weather_hourly` (validated hourly weather for all catalogue cities),
the location dimension `clean.locations` (from the city catalogue CSV) and the **gold layer**
`features.weather_features`.

Sections: 1 overview · 2 distributions · 3 correlations · 4 seasonal and daily patterns ·
5 rain sparsity · 6 completeness (missing-data pattern) · 7 features vs target · 8 preparation summary.

Run from the project root with the `venv` kernel. Figures are saved to `reports/figures/`.

In [ ]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":          # run from the project root
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import text
from common.db import get_engine

FIG = Path("reports/figures"); FIG.mkdir(parents=True, exist_ok=True)
BLUE, ORANGE = "#2a78d6", "#eb6834"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
engine = get_engine()

def save(fig, name):
    fig.tight_layout(); fig.savefig(FIG / f"{name}.png", bbox_inches="tight"); plt.show()

## 1. Overview

In [ ]:
with engine.connect() as conn:
    clean = pd.read_sql(text("SELECT * FROM clean.weather_hourly"), conn)
    locs = pd.read_sql(text("SELECT location, country, continent, climate_zone, latitude, role FROM clean.locations"), conn)
clean["ts"] = pd.to_datetime(clean["ts"], utc=True)
VARS = ["temperature_2m", "relative_humidity_2m", "dew_point_2m", "pressure_msl",
        "cloud_cover", "wind_speed_10m", "wind_direction_10m", "precipitation"]
df = clean.merge(locs, on="location", how="left")
print(f"rows: {len(df):,}   locations: {df['location'].nunique()}   from {df['ts'].min()} to {df['ts'].max()}")
display(df[VARS].describe().T.round(2))
display(df[VARS].isna().sum().rename("missing values").to_frame().T)
display(df.groupby("location").size().rename("hours").sort_values().to_frame().T)

## 2. Distributions of the eight weather variables

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, c in zip(axes.flat, VARS):
    data = df[c].dropna()
    if c == "precipitation":
        data = data[data > 0]; ax.set_yscale("log"); c += " (> 0 mm, log scale)"
    ax.hist(data, bins=50, color=BLUE)
    ax.set_title(c, loc="left", fontsize=10)
save(fig, "01_distributions")

Most hours have **no rain**, so precipitation is shown for rainy hours only. Pressure and humidity are bounded; temperature is wide because the corpus covers every climate.

## 3. Correlation heatmap

In [ ]:
corr = df[VARS].corr()
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(VARS)), VARS, rotation=45, ha="right"); ax.set_yticks(range(len(VARS)), VARS)
for i in range(len(VARS)):
    for j in range(len(VARS)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, shrink=0.8); ax.set_title("Correlation between weather variables", loc="left")
save(fig, "02_correlation_heatmap")

## 4. Seasonal and daily patterns

In [ ]:
df["month"] = df["ts"].dt.month
monthly = df.groupby(["climate_zone", "month"])["temperature_2m"].mean().unstack(0)
fig, ax = plt.subplots(figsize=(10, 5))
monthly.plot(ax=ax, marker="o")
ax.set_title("Mean temperature by month and climate zone", loc="left"); ax.set_ylabel("°C"); ax.legend(frameon=False)
save(fig, "03_monthly_by_climate_zone")

blr = df[df["location"] == "Bengaluru"].copy()
blr["local_hour"] = blr["ts"].dt.tz_convert("Asia/Kolkata").dt.hour
daily = blr.groupby("local_hour")[["temperature_2m", "relative_humidity_2m"]].mean()
fig, ax1 = plt.subplots(figsize=(10, 4.5))
ax1.plot(daily.index, daily["temperature_2m"], color=BLUE, marker="o", label="Temperature (°C)")
ax2 = ax1.twinx(); ax2.plot(daily.index, daily["relative_humidity_2m"], color=ORANGE, marker="o", label="Humidity (%)")
ax1.set_xlabel("Local hour (IST)"); ax1.set_title("Bengaluru - average daily cycle", loc="left")
fig.legend(loc="upper left", frameon=False)
save(fig, "04_bengaluru_daily_cycle")

Temperature peaks in the early afternoon while humidity is lowest - the reason **solar hour** is a feature. Seasonal curves differ by climate zone (and are reversed in the southern hemisphere) - the reason **month** and **latitude** are features.

## 5. Rain sparsity (rare-event pattern)

In [ ]:
rain = df.assign(rainy=df["precipitation"] >= 0.1).groupby("location")["rainy"].mean().sort_values() * 100
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(rain.index, rain.values, color=BLUE)
ax.set_xlabel("% of hours with rain (>= 0.1 mm)"); ax.set_title("Rain is a sparse, imbalanced target", loc="left")
save(fig, "05_rain_sparsity")
print(f"Overall rainy hours: {rain.mean():.1f}% - so rain is modelled as a probability (classification), "
      "evaluated with Brier score and AUC instead of accuracy.")

## 6. Completeness - missing-data pattern (city x month)

In [ ]:
df["ym"] = df["ts"].dt.strftime("%Y-%m")
have = df.groupby(["location", "ym"]).size().unstack(fill_value=0)
days = pd.to_datetime(have.columns + "-01").days_in_month.values
completeness = (have / (days * 24) * 100).clip(upper=100)
fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(completeness, aspect="auto", cmap="Blues", vmin=0, vmax=100)
ax.set_yticks(range(len(completeness)), completeness.index)
step = max(1, len(completeness.columns) // 12)
ax.set_xticks(range(0, len(completeness.columns), step), completeness.columns[::step], rotation=45, ha="right")
fig.colorbar(im, label="% of hours present"); ax.set_title("Hourly data completeness by city and month", loc="left")
save(fig, "06_completeness_heatmap")

Bengaluru has 2 years of history; other cities 1 year (training corpus). Partial first/last months are expected; gaps inside the range would show as light cells.

## 7. Features vs target (gold layer)

In [ ]:
with engine.connect() as conn:
    feats = pd.read_sql(text("SELECT * FROM features.weather_features WHERE feature_version = 'v3' "
                             "AND location = 'Bengaluru'"), conn)
from features.build_features import FEATURE_COLS
corr_t = feats[FEATURE_COLS + ["target_temp_24h"]].corr()["target_temp_24h"].drop("target_temp_24h").sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(corr_t.index, corr_t.values, color=[ORANGE if v < 0 else BLUE for v in corr_t.values])
ax.set_title("Correlation of each feature with temperature 24 h later (Bengaluru, v3)", loc="left")
save(fig, "07_feature_target_correlation")

## 8. Data preparation summary

| Step | What was done | Where |
|---|---|---|
| Validation | Required fields, physical ranges, no future hours | `validation/clean_weather.py` |
| Duplicates | One row per location + hour (primary key + de-duplication) | silver layer |
| Missing values | Gaps up to 3 h interpolated; rows still missing a feature dropped | `features/build_features.py` |
| Encoding | No categorical model inputs: location enters through latitude and solar hour; climate zone is used only for analysis | gold layer |
| Scaling | `StandardScaler` for linear/logistic models; tree models use raw values | `training/train.py` |
| Prepared dataset | `clean.weather_hourly` (silver) exported to `data/prepared/` for versioning | `features/export_datasets.py` |